# Set up

In [ ]:
import dotenv
from openai import OpenAI

In [4]:
dotenv.load_dotenv()

True

In [6]:
openai_client = OpenAI()

# Load documents

In [23]:
from gitsource import GithubRepositoryDataReader, chunk_documents

reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"}
)
results = reader.read()
parsed_docs = [docs.parse() for docs in results] 
chunked_documents = chunk_documents(parsed_docs, size=3000, step=1500)


# Creating the search index

In [24]:
from minsearch import AppendableIndex
index = AppendableIndex(
    text_fields={"title","descreption","content"},
    keyword_fields=["filename"]
    
)
index.fit(chunked_documents)

In [25]:
search_result =index.search( "How do I create a dahsbord in Evidently?")

# Tradition RAG

In [33]:
def search(query):
    results = index.search(query,num_results=5)
    return results

In [41]:
import json

RAG_INSTRUCTIONS = """
You're a documentation assistant. Answer the QUESTION based on the CONTEXT from our documentation.

Use only facts from the CONTEXT when answering.
If the answer isn't in the CONTEXT, say so.
"""

RAG_PROMPT_TEMPLATE = """
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()

def build_prompt(question, search_results):
    context = json.dumps(search_results, indent=2)
    return RAG_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )

In [42]:
question = "How do I create a dahsbord in Evidently?"
search_results = search(question)
user_prompt = build_prompt(question, search_results)

In [43]:
# Get the response from LLM

In [44]:
messages = [
     {"role": "system", "content": RAG_INSTRUCTIONS},
    {"role": "user", "content": user_prompt}
]
response = openai_client.responses.create(
    model = 'gpt-4o-mini',
    input = messages
    
)
    

In [45]:
print(response.output_text)

The provided context does not include specific instructions on how to create a dashboard in Evidently.


In [ ]:
# The traditional RAG couldn't find an answer because of the typo "dahsbord"

# making it agentic

In [47]:
instructions = """
You're a documentation assistant. 

Answer the user question using the documentation knowledge base

Use only facts from the knowledge base when answering.
IMPORTANT: f you cannot find the answer, inform the user.
"""

search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the documentation database for relevant results based on a query string.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "The search query to look up in the index"
            }
        },
        "required": [
            "query"
        ]
    }
}

In [48]:
messages = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": question}
]

response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=messages,
    tools=[search_tool],
)

response.usage.input_tokens

110

In [53]:
tool_call = response.output[0]
tool_call

ResponseFunctionToolCall(arguments='{"query":"create dashboard in Evidently"}', call_id='call_tcSqxjs1Gdepo8U0U1OD8R8b', name='search', type='function_call', id='fc_0950c1ab65dda0bf006ac96b61259487d2b17caa15ce4b3f95', async_=None, caller=None, namespace=None, status='completed')

In [54]:
messages.append(tool_call)
tool_call.arguments

'{"query":"create dashboard in Evidently"}'

In [55]:
arguments = json.loads(tool_call.arguments)
arguments

{'query': 'create dashboard in Evidently'}

In [56]:
search_results = search(**arguments)
search_results[:1]

[{'start': 0,
  'content': '<Note>\n  Evidently Cloud is no longer available as a SaaS product, but you can self-host the open-source Evidently Platform or continue using the Evidently library integrated with other tools.\n</Note>\n\n## 1. Create an Account\n\n- If not yet, sign up for a [free Evidently Cloud account](https://app.evidently.cloud/signup).\n- After logging in, create an **Organization** and name it.\n\n## 2. Connect from Python\n\n<Info>\n  You need this for programmatic tasks like tracing or logging local evals. Many other tasks can be done directly on the platform.\n</Info>\n\n### Get a Token\n\nClick the **Key** menu icon to open the [Token page](https://app.evidently.cloud/token). Generate and save token securely.\n\n### Install Evidently\n\n[Install](/docs/setup/installation) the Evidently Python library.\n\n```python\npip install evidently ## or pip install evidently[llm]\n```\n\n### Connect\n\nImport the cloud workspace and pass your API token to connect:\n\n```py

In [ ]:
# Continuing the Conversation

In [57]:
call_output = {
    "type": "function_call_output",
    "call_id": tool_call.call_id,
    "output": json.dumps(search_results),
}

In [58]:
messages.append(call_output)
messages

[{'role': 'system',
  'content': "\nYou're a documentation assistant. \n\nAnswer the user question using the documentation knowledge base\n\nUse only facts from the knowledge base when answering.\nIMPORTANT: f you cannot find the answer, inform the user.\n"},
 {'role': 'user', 'content': 'How do I create a dahsbord in Evidently?'},
 ResponseFunctionToolCall(arguments='{"query":"create dashboard in Evidently"}', call_id='call_tcSqxjs1Gdepo8U0U1OD8R8b', name='search', type='function_call', id='fc_0950c1ab65dda0bf006ac96b61259487d2b17caa15ce4b3f95', async_=None, caller=None, namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_tcSqxjs1Gdepo8U0U1OD8R8b',
  'output': '[{"start": 0, "content": "<Note>\\n  Evidently Cloud is no longer available as a SaaS product, but you can self-host the open-source Evidently Platform or continue using the Evidently library integrated with other tools.\\n</Note>\\n\\n## 1. Create an Account\\n\\n- If not yet, sign up for 

In [59]:
# Getting the Final Response

In [60]:
response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=messages,
    tools=[search_tool],
)

response.usage.input_tokens

3386

In [61]:
print(response.output_text)

To create a dashboard in Evidently, you have two main approaches: using the user interface (UI) or through the Python API. Here’s a basic outline for both methods:

### Using the Python API:

1. **Connect to Evidently Cloud**: Make sure you’ve connected to your Evidently Cloud account and created a project.
   
2. **Add Tabs**: To create a new tab in your dashboard:
   ```python
   project.dashboard.add_tab("Another Tab")
   ```

3. **Add Panels**: You can add different types of panels to the dashboard. Here’s how to add a simple text panel:
   ```python
   from evidently.sdk.models import PanelMetric
   from evidently.sdk.panels import DashboardPanelPlot

   project.dashboard.add_panel(
       DashboardPanelPlot(
           title="Dashboard title",
           size="full", 
           values=[],  # Leave empty for text-only
           plot_params={"plot_type": "text"},
       ),
       tab="My new tab"  # This will create the tab if it doesn't exist
   )
   ```

4. **Add Counters**: Ex

In [ ]:
# Structured_output

In [62]:
from typing import Literal
from pydantic import BaseModel, Field


class RAGResponse(BaseModel):
    """
    This model provides a structured answer with metadata about the response,
    including confidence, categorization, and follow-up suggestions.
    """

    answer: str = Field(description="The main answer to the user's question in markdown")
    found_answer: bool = Field(description="True if relevant information was found in the documentation")
    confidence: float = Field(description="Confidence score from 0.0 to 1.0 indicating how certain the answer is")
    confidence_explanation: str = Field(description="Explanation about the confidence level")
    answer_type: Literal["how-to", "explanation", "troubleshooting", "comparison", "reference"] = Field(description="The category of the answer")
    followup_questions: list[str] = Field(description="Suggested follow-up questions the user might want to ask")

In [63]:
response = openai_client.responses.parse(
    model='gpt-4o-mini',
    input=messages,
    tools=[search_tool],
    text_format=RAGResponse
)

response.usage.input_tokens

3610

In [64]:
rag_response = response.output_parsed
print(rag_response.answer)

To create a dashboard in Evidently, you can either use the user interface directly or utilize the Python API. Here’s how you can do it using the API:

### 1. Connect to Evidently Cloud
Before you start, ensure you’re connected to [Evidently Cloud](https://docs.evidently.ai/docs/setup/cloud) and have created a project.

### 2. Manage Your Dashboard
You can manage your dashboard by adding or deleting tabs and panels using the following Python code snippets:

#### Add a New Tab
```python
project.dashboard.add_tab("Your Tab Name")
```

#### Delete a Tab
```python
project.dashboard.delete_tab("Your Tab Name")
```

#### Add Panels
You can add different types of panels to display various metrics. For instance:

##### Text Panel
```python
project.dashboard.add_panel(
    DashboardPanelPlot(
        title="Your Dashboard Title",
        size="full",
        values=[], #leave empty for text
        plot_params={"plot_type": "text"},
    ),
    tab="Your Tab Name",
)
```

##### Counter Panel
To a